# TP 2 : moindres carrés, perceptron, régression logistique

Trois façons d'ajuster la même frontière, et ce que chacune fait des points sur lesquels personne
n'hésite.

## 2.0 Une décision à prendre sur 569 biopsies

569 biopsies mammaires, chacune décrite par deux mesures de la tumeur : sa **surface moyenne** et
sa **concavité moyenne**. 212 sont malignes. Il faut décider, pour chaque biopsie, entre *maligne*
et *bénigne* ; et les deux erreurs ne se paient pas au même prix.

Un classifieur linéaire lit le **signe** d'un score :

$$h_w(x) = \operatorname{sign}(w^T x), \qquad w^T x = 0 \ \text{ est la frontière} .$$

Avec deux mesures, cette frontière est une droite. Les deux colonnes sont centrées et réduites,
pour qu'un coefficient de chacune se lise sur la même échelle.

**Q1.** Deux erreurs sont possibles : annoncer *bénigne* pour une tumeur maligne, ou annoncer
*maligne* pour une tumeur bénigne. Laquelle des deux coûte le plus cher ici ? Un taux de bonnes
réponses, qui compte les deux erreurs au même prix, suffit-il à juger ce classifieur ?

**Votre réponse :**



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

np.seterr(all="ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

cancer = load_breast_cancer()
columns = [list(cancer.feature_names).index(name) for name in ("mean area", "mean concavity")]
X = StandardScaler().fit_transform(cancer.data[:, columns])
malignant = (cancer.target == 0).astype(int)       # the dataset codes 0 as malignant
t = np.where(malignant == 1, 1.0, -1.0)            # targets for the least-squares fit
area = cancer.data[:, columns[0]]

print(f"{len(X)} biopsies, dont {malignant.sum()} malignes")
print(f"surface moyenne : de {area.min():.0f} à {area.max():.0f}")


def design_matrix(X):
    return np.hstack([np.ones((len(X), 1)), X])


def boundary(ax, w, label, color, style="-"):
    """Draw the line w0 + w1 x1 + w2 x2 = 0."""
    x1 = np.array(ax.get_xlim())
    ax.plot(x1, -(w[0] + w[1] * x1) / w[2], style, color=color, lw=2, label=label)


fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(*X[malignant == 0].T, s=14, alpha=0.65, label="bénigne")
ax.scatter(*X[malignant == 1].T, s=14, alpha=0.65, label="maligne")
ax.set_xlabel("surface moyenne (centrée réduite)")
ax.set_ylabel("concavité moyenne (centrée réduite)")
ax.legend()
plt.show()

## 2.1 Les moindres carrés comme classifieur

On code les classes $+1$ et $-1$, on ajuste par moindres carrés comme s'il s'agissait d'une
régression, et on prédit le signe du score :

$$w = (X^T X)^{-1} X^T t, \qquad \hat y = \operatorname{sign}(w^T x).$$

Aucune itération, aucun taux d'apprentissage : une forme close.

**Q2.** Un patient dont la tumeur est énorme obtient un score $w^T x$ de $+5$, pour une cible de
$+1$ ; un autre, de même cible $+1$, obtient $-3$, donc du mauvais côté de la frontière. Que vaut la
contribution de chacun à la somme des carrés, et qu'en concluez-vous sur ce que cette somme
distingue ?

**Votre réponse :**



In [ ]:
def least_squares_classifier(X, t):
    # TODO: least-squares weights on the +-1 targets; np.linalg.lstsq is fine here
    ...


def classify(X, w):
    # TODO: return the predicted +-1 labels
    ...


def counts(t_true, t_predicted):
    """Missed malignant tumours, false alarms, and accuracy."""
    missed = int(np.sum((t_predicted < 0) & (t_true > 0)))
    false_alarms = int(np.sum((t_predicted > 0) & (t_true < 0)))
    return missed, false_alarms, float(np.mean(t_predicted == t_true))


w_ls = least_squares_classifier(X, t)
assert np.allclose(w_ls, np.linalg.lstsq(design_matrix(X), t, rcond=None)[0])
assert set(np.unique(classify(X, w_ls))) <= {-1.0, 1.0}

missed, false_alarms, accuracy = counts(t, classify(X, w_ls))
print(f"moindres carrés sur les 569 biopsies : justesse {accuracy:.3f}, "
      f"{missed} malignes manquées sur {int((t > 0).sum())}, {false_alarms} fausses alertes")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(*X[malignant == 0].T, s=14, alpha=0.5, label="bénigne")
ax.scatter(*X[malignant == 1].T, s=14, alpha=0.5, label="maligne")
boundary(ax, w_ls, "moindres carrés", "black")
ax.set_xlabel("surface moyenne")
ax.set_ylabel("concavité moyenne")
ax.legend()
plt.show()

## 2.2 Les cas sur lesquels personne n'hésite

Parmi ces 569 tumeurs, il y a les **10 % les plus grosses** et les **10 % les plus petites** :
celles dont le diagnostic ne fait aucun doute. On les met de côté (non pas comme des données
aberrantes, elles sont parfaitement correctes) et on demande au modèle ce qui l'intéresse
vraiment : les **454 cas restants**, ceux où la décision se joue.

Deux ajustements à comparer, tous deux **évalués sur ces 454 cas** :

- le premier entraîné sur les 569 biopsies, cas évidents inclus,
- le second entraîné sur les 454 seules.

**Q3.** Les 115 cas évidents sont loin de la frontière, du bon côté, et parfaitement classés : leur
score est donc très éloigné de la cible $\pm 1$. D'après Q2, leur contribution à la somme des carrés
est-elle petite ou grosse, et que risque-t-elle de coûter aux 454 cas du milieu ?

**Votre réponse :**



In [ ]:
obvious = (area >= np.quantile(area, 0.9)) | (area <= np.quantile(area, 0.1))
ambiguous = ~obvious
print(f"{obvious.sum()} cas évidents, {ambiguous.sum()} cas ambigus "
      f"dont {int((t[ambiguous] > 0).sum())} malignes")

w_all = least_squares_classifier(X, t)
# TODO: the same fit, on the ambiguous cases only
w_without = ...

logistic = LogisticRegression(C=1e12, max_iter=5000).fit(X, malignant)
w_logistic = np.r_[logistic.intercept_, logistic.coef_[0]]

rows = []
for label, w in (("moindres carrés, tous les cas", w_all),
                 ("moindres carrés, sans les évidents", w_without),
                 ("régression logistique, tous les cas", w_logistic)):
    missed, false_alarms, accuracy = counts(t[ambiguous], classify(X[ambiguous], w))
    rows.append({"ajustement": label, "justesse sur les 454": accuracy,
                 "malignes manquées": missed, "fausses alertes": false_alarms})

assert rows[0]["malignes manquées"] > rows[1]["malignes manquées"]
print()
print(pd.DataFrame(rows).to_string(index=False, formatters={"justesse sur les 454": "{:.3f}".format}))

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(*X[ambiguous & (malignant == 0)].T, s=14, alpha=0.5, label="bénigne, ambiguë")
ax.scatter(*X[ambiguous & (malignant == 1)].T, s=14, alpha=0.5, label="maligne, ambiguë")
ax.scatter(*X[obvious].T, s=26, facecolors="none", edgecolors="gray", lw=0.8,
           label="cas évidents (10 % plus gros, 10 % plus petits)")
boundary(ax, w_all, "moindres carrés, tous les cas", "black")
boundary(ax, w_without, "moindres carrés, sans les évidents", "black", "--")
boundary(ax, w_logistic, "régression logistique", "crimson")
ax.set_xlabel("surface moyenne")
ax.set_ylabel("concavité moyenne")
ax.legend(fontsize=8, loc="upper left")
plt.show()

**Q4.** Dans le tableau, les deux premières lignes ne diffèrent que par les 115 cas évidents,
retirés de l'entraînement pour la seconde. La colonne `justesse sur les 454` passe de 0,866 à 0,901,
soit trois points et demi, pendant que les `malignes manquées` passent de 57 à 37 et les
`fausses alertes` de 4 à 8. Lequel de ces chiffres dit ce qui a changé pour les patientes ?

**Votre réponse :**



## 2.3 Le perceptron ne corrige que ses erreurs

Le perceptron ne mesure pas l'écart entre son score et une cible. Il ne compte que ses **erreurs** :

$$E(w) = -\sum_{i \in \mathcal{M}} y^{(i)}\, w^T x^{(i)}, \qquad
  \nabla_w E = -\sum_{i \in \mathcal{M}} y^{(i)} x^{(i)},$$

où $\mathcal{M}$ est l'ensemble des points mal classés. Un point bien classé ne contribue pas,
donc les cas évidents de tout à l'heure ne peuvent rien tirer. Exemple par exemple, la règle
devient :

$$\boxed{\ \text{si } y^{(i)} w^T x^{(i)} \le 0 : \quad w \leftarrow w + \alpha\, y^{(i)} x^{(i)}
  \quad \text{sinon : rien}\ }$$

On l'essaie sur deux problèmes à deux mesures : les pétales des iris. **Setosa contre versicolor**
d'abord, puis **versicolor contre virginica**.

**Q5.** Le perceptron s'arrête quand il ne fait plus d'erreur sur une époque entière. Sur un jeu où
aucune droite ne peut séparer les deux classes, la règle encadrée trouve-t-elle toujours un point à
corriger ? Qu'en concluez-vous sur l'arrêt de la boucle ?

**Votre réponse :**



In [ ]:
iris = load_iris()
petals = iris.data[:, 2:4]         # petal length, petal width


def perceptron(X, t, alpha=0.1, n_epochs=200, seed=0):
    """Returns the weights and the number of misclassified points at each epoch."""
    rng = np.random.default_rng(seed)
    A = design_matrix(X)
    w = np.zeros(A.shape[1])
    errors_per_epoch = []
    for _ in range(n_epochs):
        errors = 0
        for i in rng.permutation(len(A)):
            # TODO: if the point is misclassified (t[i] * A[i] @ w <= 0), update w and count it
            ...
        errors_per_epoch.append(errors)
        if errors == 0:
            break
    return w, errors_per_epoch


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (a, c, label) in zip(axes, [(0, 1, "setosa contre versicolor"),
                                    (1, 2, "versicolor contre virginica")]):
    mask = np.isin(iris.target, [a, c])
    X_pair = petals[mask]
    t_pair = np.where(iris.target[mask] == a, 1.0, -1.0)
    w_perceptron, errors = perceptron(X_pair, t_pair)
    stopped = "arrêt à l'époque %d" % len(errors) if errors[-1] == 0 else \
              "%d erreurs encore à l'époque %d" % (errors[-1], len(errors))
    print(f"{label:<32} {stopped}")
    print(f"{'':<32} erreurs par époque, 12 premières : {errors[:12]}")
    ax.plot(range(1, len(errors) + 1), errors, lw=1)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("époque")
    ax.set_ylabel("points mal classés")

assert len(perceptron(petals[iris.target != 2], np.where(iris.target[iris.target != 2] == 0, 1.0, -1.0))[1]) < 200
plt.tight_layout()
plt.show()

**Q6.** À gauche, la sortie indique un arrêt à l'époque 2 ; à droite, encore 11 erreurs à l'époque
200, et la figure montre un compte qui oscille sans jamais descendre à zéro. D'après Q5, qu'est-ce
que cela dit du jeu versicolor contre virginica, et qu'est-ce qui a arrêté la boucle de ce côté ?

**Votre réponse :**



## 2.4 Une activation lisse, et une probabilité

L'échelon du perceptron ne dit que « d'un côté ou de l'autre ». La **sigmoïde** remplace la
marche par une pente, et sa sortie se lit comme une probabilité :

$$\hat y = \sigma(w^T x) = \frac{1}{1 + e^{-w^T x}} \in (0, 1), \qquad y \in \{0, 1\} .$$

La perte n'est plus quadratique mais l'**entropie croisée** :

$$J(w) = -\frac{1}{m}\sum_{i=1}^{m}\Big[y^{(i)} \log \hat y^{(i)}
   + \big(1 - y^{(i)}\big)\log\big(1 - \hat y^{(i)}\big)\Big],$$

et son gradient se simplifie en une formule d'une lisibilité frappante, le **résidu fois
l'entrée** :

$$\boxed{\ \nabla_w J = \frac{1}{m}\sum_{i=1}^{m}\big(\hat y^{(i)} - y^{(i)}\big)\, x^{(i)}\ }$$

Chaque point tire sur $w$ **à proportion de son erreur**. Un point bien classé, dont $\hat y$ est
proche de $y$, ne tire presque pas : c'est exactement ce qui manquait aux moindres carrés. Il n'y a
pas de forme close, donc on descend.

**Q7.** Pour un cas évident avec $y = 1$ et un score $w^T x$ de 9, la sigmoïde donne un $\hat y$
presque égal à 1. Que vaut alors le résidu $\hat y - y$ de la formule encadrée, et que vaudrait-il si
$\hat y$ était le score 9 lui-même, comme chez les moindres carrés ?

**Votre réponse :**



In [ ]:
def sigmoid(z):
    # TODO: the logistic function, written so that a large negative z does not overflow
    ...


def cross_entropy(X, y, w):
    p = sigmoid(design_matrix(X) @ w)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    # TODO: the mean cross-entropy of p against y
    ...


def cross_entropy_gradient(X, y, w):
    A = design_matrix(X)
    # TODO: (1 / m) * A^T (sigmoid(A w) - y)
    ...


def gradient_per_biopsy(X, y, w):
    """The same gradient, written as the mean over the m biopsies."""
    A = design_matrix(X)
    residuals = sigmoid(A @ w) - y
    # TODO: return the mean over the rows of residual_i * x_i, a vector of n + 1 numbers
    ...

w_probe = np.array([0.3, -0.7, 0.5])
assert np.isclose(sigmoid(0.0), 0.5)
assert np.isclose(sigmoid(-50), 0.0, atol=1e-12) and np.isfinite(cross_entropy(X, malignant, w_probe))
assert np.allclose(cross_entropy_gradient(X, malignant, w_probe),
                   gradient_per_biopsy(X, malignant, w_probe))

w_gd = np.zeros(3)
history = []
for step in range(4000):
    w_gd = w_gd - 0.5 * cross_entropy_gradient(X, malignant, w_gd)
    history.append(cross_entropy(X, malignant, w_gd))

print(f"entropie croisée : descente {history[-1]:.4f}, "
      f"sklearn {cross_entropy(X, malignant, w_logistic):.4f}")
print(f"poids : descente {np.round(w_gd, 3)}, sklearn {np.round(w_logistic, 3)}")

probabilities = sigmoid(design_matrix(X) @ w_gd)
rows = []
for threshold in (0.5, 0.4, 0.3, 0.2):
    predicted = np.where(probabilities >= threshold, 1.0, -1.0)
    missed, false_alarms, accuracy = counts(t, predicted)
    rows.append({"seuil": threshold, "justesse": accuracy,
                 "malignes manquées": missed, "fausses alertes": false_alarms})
print()
print(pd.DataFrame(rows).to_string(index=False, formatters={"justesse": "{:.3f}".format}))

**Q8.** Dans le tableau des seuils, le modèle est le même à chaque ligne : seul le seuil de
décision change. Les `malignes manquées` tombent de 33 au seuil 0,5 à 16 au seuil 0,3, pendant que
les `fausses alertes` montent de 20 à 35. Qu'est-ce qu'une probabilité permet de faire ici, que le
signe rendu par le perceptron ne permettrait pas ?

**Votre réponse :**



## 2.5 Trois classes, et celle du milieu qui disparaît

Trois espèces d'iris, et deux mesures : longueur du sépale et largeur du pétale. On étend les
moindres carrés au multiclasse en donnant à chaque classe **sa colonne de cibles**, valant 1 pour
ses exemples et 0 pour les autres (un codage **one-hot**), puis on prédit la classe de plus grand
score :

$$W = (X^T X)^{-1} X^T Y, \qquad \hat y = \arg\max_k \big(W^T x\big)_k .$$

La softmax fait la même chose en remplaçant l'identité par une distribution :

$$\hat y_c = \frac{\exp(w_c^T x)}{\sum_j \exp(w_j^T x)} .$$

**Q9.** Les trois espèces se rangent presque en ligne le long de la largeur de pétale : setosa, puis
versicolor, puis virginica. Le score de versicolor est une fonction linéaire de cette largeur :
peut-il être haut au milieu et bas aux deux bouts ? Qu'en concluez-vous sur l'$\arg\max$ pour cette
classe ?

**Votre réponse :**



In [ ]:
pair = [list(iris.feature_names).index(n) for n in ("sepal length (cm)", "petal width (cm)")]
X_iris = StandardScaler().fit_transform(iris.data[:, pair])
y_iris = iris.target
species = list(iris.target_names)


def one_hot(y, n_classes):
    # TODO: return the (m, n_classes) matrix with a single 1 per row
    ...


def least_squares_multiclass(X, y, n_classes):
    A = design_matrix(X)
    # TODO: solve for W, shape (n + 1, n_classes), then return it
    ...


W = least_squares_multiclass(X_iris, y_iris, 3)
assert one_hot(y_iris, 3).shape == (150, 3) and np.all(one_hot(y_iris, 3).sum(axis=1) == 1)
assert W.shape == (3, 3)

predicted_ls = np.argmax(design_matrix(X_iris) @ W, axis=1)
softmax = LogisticRegression(max_iter=5000).fit(X_iris, y_iris)
predicted_softmax = softmax.predict(X_iris)

table = pd.DataFrame({"vraies": np.bincount(y_iris, minlength=3),
                      "prédites, moindres carrés": np.bincount(predicted_ls, minlength=3),
                      "prédites, softmax": np.bincount(predicted_softmax, minlength=3)},
                     index=species)
print(table.to_string())
print(f"\njustesse : moindres carrés {np.mean(predicted_ls == y_iris):.3f}, "
      f"softmax {np.mean(predicted_softmax == y_iris):.3f}")

grid_x, grid_y = np.meshgrid(np.linspace(*np.percentile(X_iris[:, 0], [0, 100]) + [-0.5, 0.5], 400),
                             np.linspace(*np.percentile(X_iris[:, 1], [0, 100]) + [-0.5, 0.5], 400))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, regions, title in (
        (axes[0], np.argmax(design_matrix(grid) @ W, axis=1), "moindres carrés one-hot"),
        (axes[1], softmax.predict(grid), "softmax")):
    ax.contourf(grid_x, grid_y, regions.reshape(grid_x.shape), levels=[-0.5, 0.5, 1.5, 2.5],
                colors=["tab:blue", "tab:orange", "tab:green"], alpha=0.18)
    for k, name in enumerate(species):
        ax.scatter(*X_iris[y_iris == k].T, s=16, label=name)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("longueur du sépale")
    ax.set_ylabel("largeur du pétale")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Q10.** Dans le tableau, la colonne `prédites, moindres carrés` donne 4 versicolor pour 50 réelles,
et la justesse affichée est de 0,693 contre 0,960 pour la softmax. Qu'est-ce que le tableau montre
sur cette classe, que la justesse seule ne montre pas ?

**Votre réponse :**



## 2.6 Une seule règle, trois activations

Les trois algorithmes de ce TP mettent à jour $w$ avec la **même formule**, le résidu fois
l'entrée :

$$\boxed{\ w \leftarrow w + \alpha\big(y^{(i)} - \hat y^{(i)}\big) x^{(i)}\ }$$

Seule change la façon de produire $\hat y$ : l'**échelon** donne le perceptron, la **sigmoïde** la
régression logistique, l'**identité** la régression linéaire.

**Q11.** Prenez un point que le modèle classe bien : $y = 1$ et un score $w^T x$ de 9. Sous
l'échelon $\hat y$ vaut 1, sous la sigmoïde presque 1, sous l'identité 9. Que vaut le résidu
$y - \hat y$ dans les trois cas, et laquelle des trois activations continue de pousser ce point ?

**Votre réponse :**



In [ ]:
i = 0
x_i = design_matrix(X)[i]
y_i = malignant[i]

activations = {
    "échelon (perceptron)": lambda z: float(z > 0),
    "sigmoïde (logistique)": lambda z: float(sigmoid(z)),
    "identité (linéaire)": lambda z: float(z),
}

rows = []
for name, activation in activations.items():
    for w_current in (np.zeros(3), w_gd, 3 * w_gd):
        z = float(x_i @ w_current)
        y_hat = activation(z)
        # TODO: the update the rule would apply to w, with alpha = 0.1
        update = ...
        rows.append({"activation": name, "score wᵀx": z, "ŷ": y_hat,
                     "résidu y - ŷ": y_i - y_hat, "‖mise à jour‖": np.linalg.norm(update)})

assert np.isclose(rows[3]["‖mise à jour‖"], 0.1 * abs(y_i - float(sigmoid(x_i @ np.zeros(3))))
                  * np.linalg.norm(x_i))
print(f"biopsie {i} : maligne = {y_i}, entrée augmentée {np.round(x_i, 3)}")
print()
print(pd.DataFrame(rows).to_string(index=False, formatters={
    "score wᵀx": "{:7.3f}".format, "ŷ": "{:7.3f}".format,
    "résidu y - ŷ": "{:7.3f}".format, "‖mise à jour‖": "{:7.4f}".format}))

**Q12.** Dans le tableau, comparez les trois lignes où le `score wᵀx` vaut 27,280 : c'est la biopsie
0, une grosse tumeur maligne très loin du bon côté de la frontière. La norme de la mise à jour vaut
0,0000 sous l'échelon et sous la sigmoïde, et 7,887 sous l'identité. Qu'est-ce que l'identité a de
particulier, qui produit la plus grosse mise à jour du tableau sur le point le plus facile du jeu ?

**Votre réponse :**

